# 🚀 AI INTERVIEWER – Google Colab Complete Automated Deployment & Live Demo Notebook
**B.Tech CSE Final-Year Project | Django Backend + FastAPI AI Microservice + Google Gemini API (`google-genai`) + SQLite**

Run the cells below in sequence to:
1. Extract or generate the complete `ai_interviewer_platform` directory structure.
2. Install all required packages (`Django`, `fastapi`, `uvicorn`, `google-genai`, `pdfplumber`, `pypdf`, `python-docx`, `pyngrok`).
3. Configure your `GEMINI_API_KEY` securely via `getpass` or Colab Secrets.
4. Run Django database migrations (`python manage.py migrate`) and end-to-end verification tests.
5. Launch both **FastAPI (port 8001)** and **Django (port 8000)** in the background and generate an instant clickable **Live Demonstration URL**.
6. Back up your SQLite database and project ZIP to Google Drive.

In [ ]:
# STEP 1: Install Required Dependencies in Google Colab
!pip install -q Django>=4.2 fastapi uvicorn requests httpx pydantic python-multipart python-dotenv google-genai pdfplumber pypdf python-docx reportlab pyngrok

In [ ]:
# STEP 2: Extract ai_interviewer_platform.zip (Upload ai_interviewer_platform.zip to /content/ first, or it will auto-extract)
import os, zipfile
from pathlib import Path

zip_candidates = [Path('/content/ai_interviewer_platform.zip'), Path('ai_interviewer_platform.zip')]
target_dir = Path('/content/ai_interviewer_platform')
for z in zip_candidates:
    if z.exists():
        target_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z, 'r') as zf:
            zf.extractall(target_dir)
        print(f'Successfully extracted {z} to {target_dir}')
        break
else:
    print('Upload ai_interviewer_platform.zip to the Colab Files sidebar on the left and re-run this cell.')

In [ ]:
# STEP 3: Configure Environment Variables & Google Gemini API Key Securely (Never Hardcoded)
import os

from getpass import getpass

api_key = os.environ.get('GEMINI_API_KEY', '')
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('GEMINI_API_KEY') or ''
    except Exception:
        pass
if not api_key:
    api_key = getpass('Enter your Google Gemini API Key (or press Enter to use the built-in Semantic NLP Engine): ').strip()

os.environ['GEMINI_API_KEY'] = api_key
os.environ['LLM_PROVIDER'] = 'gemini'
os.environ['GEMINI_MODEL_NAME'] = 'gemini-2.5-flash'
os.environ['FASTAPI_SERVICE_URL'] = 'http://127.0.0.1:8001'
print('Environment configured. Active LLM Provider:', os.environ['LLM_PROVIDER'])

In [ ]:
# STEP 4: Run Django Database Migrations & Execute End-to-End Verification Suite
%cd /content/ai_interviewer_platform/backend
!python3 manage.py migrate
!python3 manage.py test
%cd /content/ai_interviewer_platform
!python3 tests/verify_end_to_end.py

In [ ]:
# STEP 5: Launch FastAPI AI Service (Port 8001) & Django Web Server (Port 8000) + Generate Live Demonstration URL
import subprocess, time, requests

fastapi_proc = subprocess.Popen(
    ['uvicorn', 'ai_service.main:app', '--host', '127.0.0.1', '--port', '8001'],
    cwd='/content/ai_interviewer_platform'
)
django_proc = subprocess.Popen(
    ['python3', 'manage.py', 'runserver', '0.0.0.0:8000', '--noreload'],
    cwd='/content/ai_interviewer_platform/backend'
)
time.sleep(3)
print('FastAPI Health:', requests.get('http://127.0.0.1:8001/health').json())

# 1. Native Google Colab Port Forwarding Link (No third-party account needed)
try:
    from google.colab.output import eval_js
    colab_url = eval_js('google.colab.kernel.proxyPort(8000)')
    print('\n====================================================================')
    print('🌐 CLICKABLE LIVE DEMONSTRATION URL (Colab Proxy):', colab_url)
    print('====================================================================\n')
except Exception as e:
    print('Colab proxy helper notice:', e)

In [ ]:
# STEP 6: Persistent Backup to Google Drive or Download Project ZIP
import shutil
shutil.make_archive('/content/AI_Interviewer_Backup', 'zip', '/content/ai_interviewer_platform')
print('Created backup archive: /content/AI_Interviewer_Backup.zip')
try:
    from google.colab import files
    files.download('/content/AI_Interviewer_Backup.zip')
except Exception:
    pass